# 🧮 02 - 因子计算与可视化

本 Notebook 演示：
1. 计算各类因子（动量/趋势/均值回归）
2. 因子值分布可视化
3. 多因子合成

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.aggregator import DataAggregator
from datetime import date

agg = DataAggregator()
my_stocks = ['600519.SH', '000001.SZ', '300750.SZ', '000858.SZ', '601318.SH']
bars = agg.get_bars_batch(my_stocks, date(2025, 1, 1), date.today())
print(f'✅ 数据: {len(bars)} 条, {bars.index.get_level_values(1).nunique()} 只股票')

## 1. 动量因子 (Momentum)

In [ ]:
from src.strategy.factors.momentum import MomentumFactor

mom = MomentumFactor(window=20)
mom_values = mom.compute(bars)
print(f'✅ 动量因子(20日) 计算完成: {len(mom_values)} 条')

# 最新一天的因子值
last_date = mom_values.index.get_level_values(0).max()
print(f'\n最新日期 {last_date} 各股票动量:')
print(mom_values.loc[last_date].sort_values('momentum', ascending=False))

## 2. 趋势因子 (MA交叉)

In [ ]:
from src.strategy.factors.trend import TrendFactor

trend = TrendFactor(short_window=5, long_window=20)
trend_values = trend.compute(bars)
print(f'✅ 趋势因子(5/20MA) 计算完成: {len(trend_values)} 条')

print(f'\n最新日期趋势值:')
print(trend_values.loc[last_date].sort_values('trend', ascending=False))

## 3. 因子可视化

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 动量因子时序
ax1 = axes[0]
for sym in my_stocks:
    ax1.plot(mom_values.loc[(slice(None), sym), 'momentum'], label=sym, alpha=0.7)
ax1.axhline(0, color='black', linewidth=0.5)
ax1.set_title('📈 动量因子(20日) 时序')
ax1.set_ylabel('Momentum')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 趋势因子时序
ax2 = axes[1]
for sym in my_stocks:
    ax2.plot(trend_values.loc[(slice(None), sym), 'trend'], label=sym, alpha=0.7)
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_title('📈 趋势因子(5/20MA) 时序')
ax2.set_ylabel('Trend')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. 多因子合成

In [ ]:
from src.strategy.base import CompositeFactor

# 合成因子: 动量40% + 趋势60%
composite = CompositeFactor([
    (MomentumFactor(20), 0.4),
    (TrendFactor(5, 20), 0.6),
])
composite_values = composite.compute(bars)

print(f'✅ 合成因子计算完成: {len(composite_values)} 条')
print(f'\n最新日期综合排名:')
last_scores = composite_values.loc[last_date].squeeze().sort_values(ascending=False)
for i, (sym, score) in enumerate(last_scores.items(), 1):
    print(f'  #{i} {sym}: {score:.4f}')